In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/taurusilver/proteinbase/plain_training.csv


In [2]:
import os

# 1. Force the LayerNorm module to use vanilla PyTorch instead of fast_layernorm
os.environ["LAYERNORM_TYPE"] = "torch"

# 2. Tell the underlying math modules to use vanilla PyTorch
os.environ["TRIANGLE_ATTENTION"] = "torch"
os.environ["TRIANGLE_MULTIPLICATIVE"] = "torch"

# 3. Explicitly tell PyTorch to target your P100 architecture (Compute Capability 6.0)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

print("All fallback switches forced to standard PyTorch successfully!")

All fallback switches forced to standard PyTorch successfully!


In [ ]:
!export TORCH_CUDA_ARCH_LIST="7.5"

# 2. Clear out any bad cached builds from Ninja/Pip
!rm -rf /root/.cache/pip
!rm -rf /usr/local/lib/python3.12/dist-packages/protenix/model/layer_norm/.ninja_*

# 3. Force a reinstall to trigger a fresh C++/CUDA compilation
!pip install protenix

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 21.4 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.5 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 29.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.3/517.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.1 MB/s eta 0:00:0

In [11]:
import os
import json

os.makedirs('examples', exist_ok=True)

# Corrected Schema: Protenix expects the sequence definitions 
# to use 'proteinChain' inside the sequences block.
input_data = [
    {
        "id": "target_binder_complex",
        "name": "target_binder_complex",
        "sequences": [
            {
                "proteinChain": {
                    "sequence": "MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKILSAFNTVIALLGSIVIIVMNIMIIQNYTRSTDNQAVIKDALQGIQQQIKGLADKIGTEIGPKVSLIDTSSTITIPANIGLLGSKISQSTASINENVNEKCKFTLPPLKIHECNISCPNPLPFREYRPQTEGVSNLVGLPNNICLQKTSNQILKPKLISYTLPVVGQSGTCITDPLLAMDEGYFAYSHLERIGSCSRGVSKQRIIGVGEVLDRGDEVPSLFMTNVWTPPNPNTVYHCSAVYNNEFYYVLCAVSTVGDPILNSTYWSGSLMMTRLAVKPKSNGGGYNQHQLALRSIEKGRYDKVMPYGPSGIKQGDTLYFPAVGFLVRTEFKYNDSNCPITKCQYSKPENCRLSMGIRPNSHYILRSGLLKYNLSDGENPKVVFIEISDQRLSIGSPSKIYDSLGQPVFYQASFSWDTMIKFGDVLTVNPLVVNWRNNTVISRPGQSQCPRFNTCPEICWEGVYNDAFLIDRINWISAGVFLDSNQTAENPVFTVFKDNEILYRAQLASEDTNAQKTITNCFLLKNKIWCISLVEIYDTGDNVIRPKLFAVKIPEQCT",
                    "count": 1
                }
            },
            {
                "proteinChain": {
                    "sequence": "EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLEWMGGIIPILGIANYAQKFQGRVTITTDESTSTAYMELSSLRSEDTAVYYCARGWGREQLAPHPSQYYYYYYGMDVWGQGTTVTVSSGGGGSGGGGSGGGGSEIVMTQSPGTPSLSPGERATLSCRASQSIRSTYLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGRSPSFGQGTKVEIK",
                    "count": 1
                }
            }
        ]
    }
]

with open('examples/input.json', 'w') as f:
    json.dump(input_data, f, indent=4)

print('Successfully updated examples/input.json with proper proteinChain definitions.')

Successfully updated examples/input.json with proper proteinChain definitions.


In [12]:
!protenix pred \
    -i examples/input.json \
    -o ./output \
    -d fp32 \
    --use_default_params true \
    --trimul_kernel torch \
    --triatt_kernel torch \
    --model_name protenix_base_20250630_v1.0.0

2026-06-02 07:23:54,135 [/usr/local/lib/python3.12/dist-packages/runner/batch_inference.py:835] INFO runner.batch_inference: Run infer with input=examples/input.json, out_dir=./output, sample=5
2026-06-02 07:23:54,135 [/usr/local/lib/python3.12/dist-packages/runner/batch_inference.py:863] INFO runner.batch_inference: Using default params for model protenix_base_20250630_v1.0.0: cycle=10, step=200, use_msa=True
2026-06-02 07:23:54,135 [/usr/local/lib/python3.12/dist-packages/runner/batch_inference.py:511] INFO runner.batch_inference: Will infer with 1 jsons
2026-06-02 07:23:54,413 [/usr/local/lib/python3.12/dist-packages/runner/inference.py:573] INFO runner.inference: Enforcing FP32 and torch kernels for compatibility with detected GPU (Compute Capability 7.x).
2026-06-02 07:23:54,413 [/usr/local/lib/python3.12/dist-packages/runner/batch_inference.py:413] INFO runner.batch_inference: Inference by Protenix: model_size: base, with_feature: 20250630, model_version: v1.0.0, dtype: fp32
2026

In [8]:
import json
import json
import os


def extract_binder_metrics(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None

    with open(file_path, 'r') as f:
        data = json.load(f)
    
    try:
        # In target-binder setups:
        # Index 0 = Chain A (Target)
        # Index 1 = Chain B (Binder)
        binder_ptm = data["chain_ptm"][1]
        
        # For a two-chain complex, the global 'iptm' represents the interface 
        # score between the target and the binder.
        complex_iptm = data["iptm"] 
        
        # Alternatively, you can also pull it from the chain-specific array
        # binder_iptm = data["chain_iptm"][1]

        metrics = {
            "binder_ptm": binder_ptm,
            "binder_iptm": complex_iptm
        }
        
        return metrics

    except (KeyError, IndexError) as e:
        print(f"Error parsing JSON structure: {e}")
        print("Ensure this file belongs to a multi-chain (complex) prediction.")
        return None
avg_arr = []
# Execute the extraction for the diff samples
for i in range(0,5):
    json_path = "/kaggle/working/output/target_binder_complex/seed_101/predictions/target_binder_complex_summary_confidence_sample_" + str(i) + ".json"
    binder_stats = extract_binder_metrics(json_path)
    print(binder_stats)
    avg_arr.append([binder_stats['binder_ptm'],binder_stats['binder_iptm']])
#now with a 2d array, take avg of each col 
if avg_arr:
    # --- METHOD 1: Using NumPy (Cleanest approach for 2D arrays) ---
    # axis=0 calculates the mean vertically (across rows) for each column
    column_averages = np.mean(avg_arr, axis=0)
    avg_ptm = column_averages[0]
    avg_iptm = column_averages[1]
    
    print("=== Column Averages ===")
    print(f"Average Binder PTM  : {avg_ptm:.4f}")
    print(f"Average Binder ipTM : {avg_iptm:.4f}")
else:
    print("No result")


{'binder_ptm': 0.4756133556365967, 'binder_iptm': 0.3252594769001007}
{'binder_ptm': 0.4778585731983185, 'binder_iptm': 0.3244781196117401}
{'binder_ptm': 0.46922576427459717, 'binder_iptm': 0.32534393668174744}
{'binder_ptm': 0.46329009532928467, 'binder_iptm': 0.3247281014919281}
{'binder_ptm': 0.46752607822418213, 'binder_iptm': 0.3231144845485687}
=== Column Averages ===
Average Binder PTM  : 0.4707
Average Binder ipTM : 0.3246


In [10]:
import pandas as pd
data = pd.read_csv("/kaggle/input/datasets/taurusilver/proteinbase/plain_training.csv")
data

,sequence,target_sequence,value
0,EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
1,EVKLEESGGGLVQPGGSMKLSCVASGFSFSYYWMNWVRQSPEKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
2,KSIVLEPIYWNSSNSKFLPGQGLVLYPQIGDKLDIICPKVDSKTVG...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
3,EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
4,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
...,...,...,...
523,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Medium
524,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Medium
525,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Strong
526,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Weak


In [ ]:
binder = data['sequence']
target = data['target_sequence']


In [ ]:
import os
import json
import subprocess
import pandas as pd
import numpy as np

def extract_binder_metrics(file_path):
    """Extracts pTM and ipTM metrics from Protenix summary JSON."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}")
        return None

    with open(file_path, 'r') as f:
        data = json.load(f)
    
    try:
        # Index 0 = Chain A (Target), Index 1 = Chain B (Binder)
        binder_ptm = data["chain_ptm"][1]
        complex_iptm = data["iptm"] 
        
        return {"binder_ptm": binder_ptm, "binder_iptm": complex_iptm}
    except (KeyError, IndexError) as e:
        print(f"Error parsing JSON structure for {file_path}: {e}")
        return None

def screen_complexes_with_protenix(dataset, output_csv="protenix_results.csv"):
    """
    Loops through data, runs Protenix, averages sample metrics, and saves to CSV.
    
    Parameters:
    dataset (list of dicts): Must contain 'target_sequence' and 'sequence' (binder) keys.
    output_csv (str): Filename for the saved CSV results.
    """
    results_list = []
    os.makedirs('examples', exist_ok=True)

    for index, entry in enumerate(dataset):
        binder_seq = entry.get('sequence')
        target_seq = entry.get('target_sequence')
        
        # Unique identifier for each complex run
        complex_id = f"complex_{index}"
        print(f"\n--- Processing {complex_id} ---")
        
        # 1. Format and write the input JSON for Protenix
        input_data = [
            {
                "id": complex_id,
                "name": complex_id,
                "sequences": [
                    {"proteinChain": {"sequence": target_seq, "count": 1}}, # Chain A
                    {"proteinChain": {"sequence": binder_seq, "count": 1}}  # Chain B
                ]
            }
        ]
        
        input_json_path = 'examples/input.json'
        with open(input_json_path, 'w') as f:
            json.dump(input_data, f, indent=4)
            
        # 2. Execute Protenix Prediction using subprocess
        protenix_cmd = [
            "protenix", "pred",
            "-i", input_json_path,
            "-o", "./output",
            "-d", "fp32",
            "--use_default_params", "true",
            "--trimul_kernel", "torch",
            "--triatt_kernel", "torch",
            "--model_name", "protenix_base_20250630_v1.0.0"
        ]
        
        try:
            print(f"Running Protenix inference for {complex_id}...")
            subprocess.run(protenix_cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"Protenix failed for {complex_id}. Skipping metrics extraction. Error: {e}")
            continue

        # 3. Gather and average metrics from the 5 generated samples
        avg_arr = []
        for i in range(5):
            json_path = f"./output/{complex_id}/seed_101/predictions/{complex_id}_summary_confidence_sample_{i}.json"
            binder_stats = extract_binder_metrics(json_path)
            
            if binder_stats:
                avg_arr.append([binder_stats['binder_ptm'], binder_stats['binder_iptm']])
        
        # 4. Compute metrics and append to results list
        if avg_arr:
            column_averages = np.mean(avg_arr, axis=0)
            avg_ptm = column_averages[0]
            avg_iptm = column_averages[1]
        else:
            avg_ptm, avg_iptm = None, None
            
        print(f"Results for {complex_id} -> Avg pTM: {avg_ptm}, Avg ipTM: {avg_iptm}")
        
        results_list.append({
            "complex_id": complex_id,
            "target_sequence": target_seq,
            "binder_sequence": binder_seq,
            "avg_binder_ptm": avg_ptm,
            "avg_binder_iptm": avg_iptm
        })

    # 5. Save all aggregated results to CSV
    df_results = pd.DataFrame(results_list)
    df_results.to_csv(output_csv, index=False)
    print(f"\nSuccessfully saved all screening results to {output_csv}")
    return df_results

In [ ]:

    
# Run the screening function
final_df = screen_complexes_with_protenix(data, output_csv="screening_output.csv")